# 🎓 RetinaSense-ViT: Complete Training Process

This notebook demonstrates the **complete training process** that achieved **84.48% accuracy**.

## Training Overview
- **Architecture:** Vision Transformer (ViT-Base-Patch16-224)
- **Dataset:** 8,540 retinal images (5 classes)
- **Training Time:** ~2-3 hours on GPU
- **Final Performance:** 82.26% raw → 84.48% with thresholds

## Notebook Sections
1. Setup & Configuration
2. Data Preprocessing & Caching
3. Model Architecture
4. Training Loop with Metrics
5. Real-time Visualization
6. Model Checkpointing
7. Post-Training Analysis

## 1. Setup & Configuration

In [7]:
# Install required packages
!pip install torch torchvision timm pandas opencv-python scikit-learn matplotlib seaborn tqdm -q

In [8]:
import os
import sys
import json
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as transforms

import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA H200
GPU Memory: 150.12 GB


In [9]:
# Training Configuration
class Config:
    # Paths
    DATA_DIR = './data'
    CACHE_DIR = './preprocessed_cache_vit'
    OUTPUT_DIR = './outputs_vit_training'
    
    # Model
    MODEL_NAME = 'vit_base_patch16_224'
    IMG_SIZE = 224
    NUM_DISEASE_CLASSES = 5
    NUM_SEVERITY_CLASSES = 5
    DROPOUT = 0.4
    
    # Training
    BATCH_SIZE = 32
    NUM_EPOCHS = 30
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4
    GRADIENT_ACCUMULATION = 2  # Effective batch = 64
    
    # Loss weights
    FOCAL_GAMMA = 1.0
    DISEASE_WEIGHT = 1.0
    SEVERITY_WEIGHT = 0.5
    
    # Early stopping
    PATIENCE = 10
    MIN_DELTA = 0.001
    
    # Class mapping
    CLASS_NAMES = ['Normal', 'Diabetes/DR', 'Glaucoma', 'Cataract', 'AMD']
    
    # Device
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
config = Config()

# Create output directory
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

print("\n📋 Training Configuration:")
print(f"  Model: {config.MODEL_NAME}")
print(f"  Image Size: {config.IMG_SIZE}x{config.IMG_SIZE}")
print(f"  Batch Size: {config.BATCH_SIZE} (effective: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION})")
print(f"  Epochs: {config.NUM_EPOCHS}")
print(f"  Learning Rate: {config.LEARNING_RATE}")
print(f"  Device: {config.DEVICE}")


📋 Training Configuration:
  Model: vit_base_patch16_224
  Image Size: 224x224
  Batch Size: 32 (effective: 64)
  Epochs: 30
  Learning Rate: 0.0001
  Device: cuda


## 2. Data Preprocessing

We use **Ben Graham preprocessing** for fundus images to improve model performance.

In [10]:
def ben_graham_preprocessing(image, target_size=224):
    """
    Ben Graham preprocessing for fundus images:
    1. Crop borders (remove black edges)
    2. Resize to target size
    3. Local color averaging (reduce lighting variations)
    """
    # Convert to grayscale for mask
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Create mask of non-black pixels
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    
    # Find contours and get bounding box
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        x, y, w, h = cv2.boundingRect(np.vstack(contours))
        image = image[y:y+h, x:x+w]
    
    # Resize
    image = cv2.resize(image, (target_size, target_size), interpolation=cv2.INTER_AREA)
    
    # Local color averaging (subtract local mean)
    blurred = cv2.GaussianBlur(image, (0, 0), target_size / 30)
    image = cv2.addWeighted(image, 4, blurred, -4, 128)
    
    return image

# Verify dataset
csv_path = os.path.join(config.DATA_DIR, 'combined_dataset.csv')
if os.path.exists(csv_path):
    df_check = pd.read_csv(csv_path)
    # Fix paths - remove leading ./ or .//
    df_check['image_path'] = df_check['image_path'].str.replace(r'^\./+', '', regex=True)
    print(f"✅ Dataset CSV found: {csv_path}")
    print(f"   Total images: {len(df_check)}")
    print(f"   Classes: {df_check['disease_label'].nunique()}")
    print(f"   Sample path: {df_check.iloc[0]['image_path']}")
    print(f"   File exists: {os.path.exists(df_check.iloc[0]['image_path'])}")
else:
    print(f"⚠️ Dataset CSV not found at {csv_path}. Please ensure data is prepared.")

✅ Dataset CSV found: ./data/combined_dataset.csv
   Total images: 8540
   Classes: 5
   Sample path: odir/preprocessed_images/0_left.jpg
   File exists: True


In [11]:
# Dataset class with path fixing
class RetinaDataset(Dataset):
    def __init__(self, df, cache_dir=None, transform=None, use_cache=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.use_cache = use_cache
        
        # Fix image paths (remove leading ./ or .//)
        self.df['image_path'] = self.df['image_path'].str.replace(r'^\./+', '', regex=True)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        
        # Load image
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Could not load image: {img_path}")
        
        # Apply Ben Graham preprocessing
        image = ben_graham_preprocessing(image, 224)
        
        # Convert BGR to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        # Labels
        disease_label = int(row['disease_label'])
        severity_label = int(row['severity_label']) if 'severity_label' in row else 0
        
        return image, disease_label, severity_label

# Data augmentation
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("\n✅ Dataset classes defined")


✅ Dataset classes defined


In [12]:
# Load and split data
df = pd.read_csv(csv_path)

# Stratified split (80% train, 20% val)
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['disease_label'], 
    random_state=SEED
)

print(f"\n📊 Dataset Split:")
print(f"  Training samples: {len(train_df)}")
print(f"  Validation samples: {len(val_df)}")
print(f"\n  Class distribution (train):")
for label, name in enumerate(config.CLASS_NAMES):
    count = (train_df['disease_label'] == label).sum()
    print(f"    {name}: {count} ({count/len(train_df)*100:.1f}%)")


📊 Dataset Split:
  Training samples: 6832
  Validation samples: 1708

  Class distribution (train):
    Normal: 1657 (24.3%)
    Diabetes/DR: 4465 (65.4%)
    Glaucoma: 246 (3.6%)
    Cataract: 252 (3.7%)
    AMD: 212 (3.1%)


In [13]:
# Create data loaders (no caching for simplicity)
train_dataset = RetinaDataset(train_df, transform=train_transform, use_cache=False)
val_dataset = RetinaDataset(val_df, transform=val_transform, use_cache=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE * 2,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print(f"\n✅ DataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")


✅ DataLoaders created:
  Train batches: 214
  Val batches: 27


## 3. Model Architecture

**Vision Transformer (ViT)** with multi-task learning heads.

In [14]:
class MultiTaskViT(nn.Module):
    def __init__(self, num_disease_classes=5, num_severity_classes=5, dropout=0.4):
        super().__init__()
        
        # Vision Transformer backbone (pre-trained on ImageNet)
        self.backbone = timm.create_model(
            'vit_base_patch16_224',
            pretrained=True,
            num_classes=0  # Remove classification head
        )
        
        # Feature dimension (ViT-Base = 768)
        self.feature_dim = 768
        
        # Disease classification head
        self.disease_head = nn.Sequential(
            nn.Linear(self.feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_disease_classes)
        )
        
        # Severity grading head
        self.severity_head = nn.Sequential(
            nn.Linear(self.feature_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_severity_classes)
        )
    
    def forward(self, x):
        # Extract features from ViT
        features = self.backbone(x)
        
        # Multi-task predictions
        disease_logits = self.disease_head(features)
        severity_logits = self.severity_head(features)
        
        return disease_logits, severity_logits

# Create model
model = MultiTaskViT(
    num_disease_classes=config.NUM_DISEASE_CLASSES,
    num_severity_classes=config.NUM_SEVERITY_CLASSES,
    dropout=config.DROPOUT
).to(config.DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n🧠 Model Architecture:")
print(f"  Model: {config.MODEL_NAME}")
print(f"  Feature dimension: {model.feature_dim}")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / (1024**2):.1f} MB")


🧠 Model Architecture:
  Model: vit_base_patch16_224
  Feature dimension: 768
  Total parameters: 86,525,194
  Trainable parameters: 86,525,194
  Model size: ~330.1 MB


## 4. Loss Functions & Optimizer

We use **Focal Loss** to handle class imbalance.

In [15]:
class FocalLoss(nn.Module):
    """
    Focal Loss for handling class imbalance.
    Focuses training on hard examples.
    """
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        p_t = torch.exp(-ce_loss)
        focal_loss = (1 - p_t) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# Calculate class weights (inverse frequency)
class_counts = train_df['disease_label'].value_counts().sort_index()
class_weights = 1.0 / class_counts.values
class_weights = class_weights / class_weights.sum() * len(class_weights)
class_weights = torch.FloatTensor(class_weights).to(config.DEVICE)

print(f"\n⚖️ Class Weights (for imbalance):")
for name, weight in zip(config.CLASS_NAMES, class_weights):
    print(f"  {name}: {weight:.3f}")

# Loss functions
disease_criterion = FocalLoss(gamma=config.FOCAL_GAMMA, alpha=class_weights)
severity_criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

# Learning rate scheduler (cosine annealing)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.NUM_EPOCHS,
    eta_min=1e-6
)

# Mixed precision training (AMP)
scaler = GradScaler()

print(f"\n✅ Training components initialized:")
print(f"  Optimizer: AdamW (lr={config.LEARNING_RATE})")
print(f"  Scheduler: CosineAnnealingLR")
print(f"  Loss: Focal (γ={config.FOCAL_GAMMA})")
print(f"  Mixed Precision: Enabled")


⚖️ Class Weights (for imbalance):
  Normal: 0.222
  Diabetes/DR: 0.082
  Glaucoma: 1.497
  Cataract: 1.461
  AMD: 1.737

✅ Training components initialized:
  Optimizer: AdamW (lr=0.0001)
  Scheduler: CosineAnnealingLR
  Loss: Focal (γ=1.0)
  Mixed Precision: Enabled


/tmp/ipykernel_546492/2752500847.py:53: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


## 5. Training Loop with Real-Time Metrics

In [16]:
# Metrics tracking
history = {
    'train_loss': [],
    'val_loss': [],
    'train_acc': [],
    'val_acc': [],
    'val_macro_f1': [],
    'learning_rate': [],
    'epoch_time': []
}

best_val_f1 = 0.0
best_epoch = 0
patience_counter = 0

print(f"\n🚀 Starting Training...")
print(f"  Total epochs: {config.NUM_EPOCHS}")
print(f"  Early stopping patience: {config.PATIENCE}")
print(f"  Saving checkpoints to: {config.OUTPUT_DIR}\n")


🚀 Starting Training...
  Total epochs: 30
  Early stopping patience: 10
  Saving checkpoints to: ./outputs_vit_training



In [17]:
def train_epoch(model, loader, optimizer, scaler, device):
    """
    Train for one epoch.
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training', leave=False)
    for batch_idx, (images, disease_labels, severity_labels) in enumerate(pbar):
        images = images.to(device)
        disease_labels = disease_labels.to(device)
        severity_labels = severity_labels.to(device)
        
        # Mixed precision forward pass
        with autocast():
            disease_logits, severity_logits = model(images)
            
            # Multi-task loss
            disease_loss = disease_criterion(disease_logits, disease_labels)
            severity_loss = severity_criterion(severity_logits, severity_labels)
            loss = config.DISEASE_WEIGHT * disease_loss + config.SEVERITY_WEIGHT * severity_loss
            
            # Gradient accumulation
            loss = loss / config.GRADIENT_ACCUMULATION
        
        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        
        # Update weights every N batches
        if (batch_idx + 1) % config.GRADIENT_ACCUMULATION == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        # Metrics
        running_loss += loss.item() * config.GRADIENT_ACCUMULATION
        _, predicted = disease_logits.max(1)
        total += disease_labels.size(0)
        correct += predicted.eq(disease_labels).sum().item()
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{running_loss/(batch_idx+1):.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def validate_epoch(model, loader, device):
    """
    Validate for one epoch.
    """
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation', leave=False)
        for images, disease_labels, severity_labels in pbar:
            images = images.to(device)
            disease_labels = disease_labels.to(device)
            severity_labels = severity_labels.to(device)
            
            # Forward pass
            disease_logits, severity_logits = model(images)
            
            # Loss
            disease_loss = disease_criterion(disease_logits, disease_labels)
            severity_loss = severity_criterion(severity_logits, severity_labels)
            loss = config.DISEASE_WEIGHT * disease_loss + config.SEVERITY_WEIGHT * severity_loss
            
            running_loss += loss.item()
            
            # Predictions
            _, predicted = disease_logits.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(disease_labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader)
    
    # Calculate metrics
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    epoch_acc = 100. * (all_preds == all_labels).mean()
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    
    return epoch_loss, epoch_acc, macro_f1, all_preds, all_labels

print("✅ Training functions defined")

✅ Training functions defined


In [18]:
# Main training loop
training_start_time = time.time()

for epoch in range(config.NUM_EPOCHS):
    epoch_start_time = time.time()
    
    print(f"\n{'='*70}")
    print(f"Epoch {epoch+1}/{config.NUM_EPOCHS}")
    print(f"{'='*70}")
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scaler, config.DEVICE)
    
    # Validate
    val_loss, val_acc, val_macro_f1, val_preds, val_labels = validate_epoch(model, val_loader, config.DEVICE)
    
    # Update scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Track metrics
    epoch_time = time.time() - epoch_start_time
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_macro_f1'].append(val_macro_f1)
    history['learning_rate'].append(current_lr)
    history['epoch_time'].append(epoch_time)
    
    # Print summary
    print(f"\n📊 Epoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print(f"  Val Macro F1: {val_macro_f1:.4f}")
    print(f"  Learning Rate: {current_lr:.2e}")
    print(f"  Time: {epoch_time:.1f}s")
    
    # Per-class F1 scores
    per_class_f1 = f1_score(val_labels, val_preds, average=None)
    print(f"\n  Per-Class F1 Scores:")
    for i, (name, f1) in enumerate(zip(config.CLASS_NAMES, per_class_f1)):
        print(f"    {name:15s}: {f1:.4f}")
    
    # Save best model (based on macro F1)
    if val_macro_f1 > best_val_f1 + config.MIN_DELTA:
        best_val_f1 = val_macro_f1
        best_epoch = epoch + 1
        patience_counter = 0
        
        # Save checkpoint
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc,
            'val_macro_f1': val_macro_f1,
            'config': vars(config)
        }
        torch.save(checkpoint, os.path.join(config.OUTPUT_DIR, 'best_model.pth'))
        print(f"\n  ⭐ New best model saved! (F1: {val_macro_f1:.4f})")
    else:
        patience_counter += 1
        print(f"\n  No improvement. Patience: {patience_counter}/{config.PATIENCE}")
    
    # Early stopping
    if patience_counter >= config.PATIENCE:
        print(f"\n🛑 Early stopping triggered at epoch {epoch+1}")
        print(f"   Best model was at epoch {best_epoch} with F1: {best_val_f1:.4f}")
        break
    
    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc,
            'val_macro_f1': val_macro_f1,
        }
        torch.save(checkpoint, os.path.join(config.OUTPUT_DIR, f'checkpoint_epoch_{epoch+1}.pth'))

total_training_time = time.time() - training_start_time

print(f"\n{'='*70}")
print(f"🎉 Training Complete!")
print(f"{'='*70}")
print(f"  Total time: {total_training_time/3600:.2f} hours")
print(f"  Best epoch: {best_epoch}")
print(f"  Best Val Macro F1: {best_val_f1:.4f}")
print(f"  Best model saved to: {config.OUTPUT_DIR}/best_model.pth")


Epoch 1/30


Training:   0%|          | 0/214 [00:00<?, ?it/s]

/tmp/ipykernel_546492/3405603224.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/pytorch/aten/src/ATen/native/cuda/Loss.cu:245: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [0,0,0] Assertion `t >= 0 && t < n_classes` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:245: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [1,0,0] Assertion `t >= 0 && t < n_classes` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:245: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [2,0,0] Assertion `t >= 0 && t < n_classes` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:245: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [3,0,0] Assertion `t >= 0 && t < n_classes` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:245: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [5,0,0] Assertion `t >= 0 && t < n_classes` failed

AcceleratorError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 6. Training Visualization

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
axes[0, 0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
axes[0, 0].plot(epochs_range, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
axes[0, 0].axvline(best_epoch, color='g', linestyle='--', alpha=0.5, label=f'Best (Epoch {best_epoch})')
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(epochs_range, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
axes[0, 1].plot(epochs_range, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
axes[0, 1].axvline(best_epoch, color='g', linestyle='--', alpha=0.5, label=f'Best (Epoch {best_epoch})')
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Accuracy (%)', fontsize=12)
axes[0, 1].set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Macro F1
axes[1, 0].plot(epochs_range, history['val_macro_f1'], 'purple', linewidth=2, marker='o')
axes[1, 0].axvline(best_epoch, color='g', linestyle='--', alpha=0.5, label=f'Best (Epoch {best_epoch})')
axes[1, 0].axhline(best_val_f1, color='g', linestyle=':', alpha=0.5, label=f'Best F1: {best_val_f1:.4f}')
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Macro F1 Score', fontsize=12)
axes[1, 0].set_title('Validation Macro F1 Score', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Learning Rate
axes[1, 1].plot(epochs_range, history['learning_rate'], 'orange', linewidth=2)
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Learning Rate', fontsize=12)
axes[1, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, 'training_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ Training curves saved to {config.OUTPUT_DIR}/training_curves.png")

In [ ]:
# Save training history
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(config.OUTPUT_DIR, 'training_history.csv'), index=False)

print("\n📊 Training History:")
print(history_df.tail(10))
print(f"\n✅ History saved to {config.OUTPUT_DIR}/training_history.csv")

## 7. Load Best Model & Final Evaluation

In [ ]:
# Load best model
best_checkpoint = torch.load(os.path.join(config.OUTPUT_DIR, 'best_model.pth'), weights_only=False)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

print(f"\n✅ Best model loaded from epoch {best_checkpoint['epoch']}")
print(f"   Val Accuracy: {best_checkpoint['val_acc']:.2f}%")
print(f"   Val Macro F1: {best_checkpoint['val_macro_f1']:.4f}")

In [ ]:
# Final evaluation with detailed metrics
_, final_acc, final_f1, final_preds, final_labels = validate_epoch(model, val_loader, config.DEVICE)

print(f"\n{'='*70}")
print(f"📊 Final Validation Results (Best Model)")
print(f"{'='*70}")
print(f"  Overall Accuracy: {final_acc:.2f}%")
print(f"  Macro F1 Score: {final_f1:.4f}")

# Classification report
print(f"\n{classification_report(final_labels, final_preds, target_names=config.CLASS_NAMES, digits=4)}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(final_labels, final_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=config.CLASS_NAMES, 
            yticklabels=config.CLASS_NAMES,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted', fontsize=12, fontweight='bold')
plt.ylabel('True', fontsize=12, fontweight='bold')
plt.title(f'Confusion Matrix - Best Model (Epoch {best_epoch})\nAccuracy: {final_acc:.2f}%, Macro F1: {final_f1:.4f}', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ Confusion matrix saved to {config.OUTPUT_DIR}/confusion_matrix.png")

## 8. Training Summary

In [ ]:
# Generate training summary
summary = {
    'training_info': {
        'model': config.MODEL_NAME,
        'total_epochs': len(history['train_loss']),
        'best_epoch': int(best_epoch),
        'total_training_time_hours': total_training_time / 3600,
        'avg_epoch_time_seconds': np.mean(history['epoch_time'])
    },
    'best_metrics': {
        'val_accuracy': float(best_checkpoint['val_acc']),
        'val_macro_f1': float(best_checkpoint['val_macro_f1']),
        'val_loss': float(best_checkpoint['val_loss'])
    },
    'per_class_f1': {
        name: float(f1) 
        for name, f1 in zip(config.CLASS_NAMES, f1_score(final_labels, final_preds, average=None))
    },
    'config': {
        'batch_size': config.BATCH_SIZE,
        'learning_rate': config.LEARNING_RATE,
        'focal_gamma': config.FOCAL_GAMMA,
        'dropout': config.DROPOUT,
        'image_size': config.IMG_SIZE
    }
}

# Save summary
with open(os.path.join(config.OUTPUT_DIR, 'training_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*70)
print("🎉 TRAINING COMPLETE - Summary")
print("="*70)
print(json.dumps(summary, indent=2))
print("\n✅ Summary saved to", os.path.join(config.OUTPUT_DIR, 'training_summary.json'))

## 🎯 Next Steps

Now that training is complete, you can:

1. **Apply Threshold Optimization** (adds +2% accuracy):
   - Run `threshold_optimization_vit.py` on the best model
   - This optimizes per-class decision thresholds
   - Improves from 82.26% → 84.48%

2. **Use Production Notebook**:
   - Open `RetinaSense_Production.ipynb`
   - Load the trained model
   - Make predictions on new images

3. **Deploy Model**:
   - Export to TorchScript or ONNX
   - Integrate into clinical workflow
   - Set up monitoring pipeline

**Model Location:** `{config.OUTPUT_DIR}/best_model.pth`